In [1]:
WORK = "/kaggle/working"      # change this if your writable dir differs
N_THREADS = 4                 # match your CPU core count

import os
os.makedirs(WORK, exist_ok=True)
os.environ["WORK"] = WORK
print(WORK)


/kaggle/working


In [2]:
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu
!pip install -q gguf git+https://github.com/Aratako/MioCodec
!git clone --depth 1 https://github.com/ggml-org/llama.cpp $WORK/llama.cpp


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.9/23.9 MB 57.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 3.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 800.2/800.2 kB 19.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.0/158.0 kB 10.8 MB/s eta 0:00:00
Cloning into '/kaggle/working/llama.cpp'...
remote: Enumerating objects: 3859, done.
remote: Counting objects: 100% (3859/3859), done.
remote: Compressing objects: 100% (3149/3149), done.
remote: Total 3859 (delta 687), reused 2671 (delta 628), pack-reused 0 (from 0)
Receiving objects: 100% (3859/3859), 35.57 MiB | 18.84 MiB/s, done.
Resolving deltas: 100% (687/687), done.


In [3]:
from huggingface_hub import snapshot_download

snapshot_download("SPRINGLab/Indic-Mio", local_dir=f"{WORK}/indic-mio-hf")


Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

'/kaggle/working/indic-mio-hf'

In [4]:
!python $WORK/llama.cpp/convert_hf_to_gguf.py $WORK/indic-mio-hf --outfile $WORK/indic-mio-f16.gguf --outtype f16


INFO:hf-to-gguf:Loading model: indic-mio-hf
INFO:hf-to-gguf:Model architecture: Qwen3ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.bfloat16 --> F16, shape = {1024, 164480}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.bfloat16 --> F32, shape = {1024}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.bfloat16 --> F16, shape = {3072, 1024}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.bfloat16 --> F16, shape = {1024, 3072}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.bfloat16 --> F16, shape = {1024, 3072}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,     torch.bfloat16 --> F32, shape = {1024}
INFO:hf-to-gguf:blk.0.attn_k_norm.weight,  torch.bfloat16 --> F32, shape = {128}
INFO:hf-to-gguf:blk.0.attn_k.weight,       torch.bfloat16 --> F16, shape = {1024, 1024}
INFO:hf-to-gguf:blk.0.attn_output.weight,  torch.b

In [5]:
!cmake -B $WORK/llama.cpp/build -S $WORK/llama.cpp -DCMAKE_BUILD_TYPE=Release -DGGML_NATIVE=ON -DLLAMA_CURL=OFF
!cmake --build $WORK/llama.cpp/build --target llama-quantize llama-bench -j4


-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- llama.cpp version: 0.3.0-dev
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Found Ope

In [6]:
!$WORK/llama.cpp/build/bin/llama-quantize $WORK/indic-mio-f16.gguf $WORK/indic-mio-q8.gguf Q8_0
!$WORK/llama.cpp/build/bin/llama-quantize $WORK/indic-mio-f16.gguf $WORK/indic-mio-q4.gguf Q4_K_M
!ls -lh $WORK/*.gguf


version: 0.3.0-dev (build 1, commit 2cdae80)
built with GNU 11.4.0 for Linux x86_64
llama_quantize: quantizing '/kaggle/working/indic-mio-f16.gguf' to '/kaggle/working/indic-mio-q8.gguf' as Q8_0
llama_model_loader: loaded meta data with 42 key-value pairs and 310 tensors from /kaggle/working/indic-mio-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen3
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Indic-Mio
llama_model_loader: - kv   3:                         general.size_label str              = 609M
llama_model_loader: - kv   4:                            general.license str              = apache-2.0
llama_model_loader: - kv   5:                   general.base_mode

In [7]:
!$WORK/llama.cpp/build/bin/llama-bench -m $WORK/indic-mio-f16.gguf -m $WORK/indic-mio-q8.gguf -m $WORK/indic-mio-q4.gguf -p 32 -n 128 -t 2,4


| model                          |       size |     params | backend    | threads |            test |                  t/s |
| ------------------------------ | ---------: | ---------: | ---------- | ------: | --------------: | -------------------: |
| qwen3 0.6B F16                 |   1.13 GiB |   608.89 M | CPU        |       2 |            pp32 |         73.32 ± 1.08 |
| qwen3 0.6B F16                 |   1.13 GiB |   608.89 M | CPU        |       2 |           tg128 |         14.30 ± 0.27 |
| qwen3 0.6B F16                 |   1.13 GiB |   608.89 M | CPU        |       4 |            pp32 |         90.74 ± 1.44 |
| qwen3 0.6B F16                 |   1.13 GiB |   608.89 M | CPU        |       4 |           tg128 |         16.56 ± 0.27 |
| qwen3 0.6B Q8_0                | 617.16 MiB |   608.89 M | CPU        |       2 |            pp32 |         75.40 ± 0.34 |
| qwen3 0.6B Q8_0                | 617.16 MiB |   608.89 M | CPU        |       2 |           tg128 |         19.61 ± 0.34 |


In [8]:
import glob, torch
from miocodec import MioCodecModel, load_audio
from huggingface_hub import hf_hub_download

torch.set_num_threads(N_THREADS)

codec = MioCodecModel.from_pretrained("Aratako/MioCodec-25Hz-24kHz").eval()
SR = codec.config.sample_rate                     # 24000

# use your own clip if one is mounted, else fall back to a sample from the model repo
found = glob.glob("/kaggle/input/**/*.ogg", recursive=True) \
      + glob.glob("/kaggle/input/**/*.wav", recursive=True)
ref_path = found[0] if found else hf_hub_download("SPRINGLab/Indic-MIO", "samples/sample1.wav")
print("reference voice:", ref_path)

voice = codec.encode(load_audio(ref_path, SR), return_content=False).global_embedding


[2026-08-31 09:22:32,961] WARNING miocodec: FlashAttention is not installed. Falling back to PyTorch SDPA implementation. There is no warranty that the model will work correctly.


config.yaml: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/523M [00:00<?, ?B/s]

Downloading: "https://download.pytorch.org/torchaudio/models/wavlm_base_plus.pth" to /root/.cache/torch/hub/checkpoints/wavlm_base_plus.pth


100%|██████████| 360M/360M [00:01<00:00, 286MB/s]  
[2026-08-31 09:22:49,217] INFO miocodec: Loaded weights from safetensors file: /root/.cache/huggingface/hub/models--Aratako--MioCodec-25Hz-24kHz/snapshots/3a737f0de2c6324cb2fe40c1fbd1056c7add423d/model.safetensors


reference voice: /kaggle/input/datasets/axamine/henil-audio/WhatsApp Ptt 2026-08-19 at 12.46.33.wav


In [9]:
import time, soundfile as sf
from llama_cpp import Llama
from transformers import AutoTokenizer
from IPython.display import Audio, display

SPEECH_OFFSET, SPEECH_END = 151669, 164469

SENTENCES = [
    "नमस्ते, आप कैसे हैं? <happy>",
    "This is a test sentence in English.",
    "प्लान तो बढ़िया है, but wait... <confused>",
]

tok = AutoTokenizer.from_pretrained("SPRINGLab/Indic-Mio")
summary = {}

for name in ["q8", "q4"]:
    llm = Llama(f"{WORK}/indic-mio-{name}.gguf", n_ctx=2048,
                n_threads=N_THREADS, seed=42, verbose=False)
    print(f"\n===== {name.upper()} =====")
    rtfs = []

    for i, text in enumerate(SENTENCES):
        prompt = tok.apply_chat_template([{"role": "user", "content": text}],
                                         tokenize=False, add_generation_prompt=True)
        ids = llm.tokenize(prompt.encode(), add_bos=False, special=True)

        llm.reset()
        start, codes = time.time(), []
        for t in llm.generate(ids, temp=0.9, top_p=0.9):
            if t == llm.token_eos() or len(codes) >= 512:
                break
            if SPEECH_OFFSET <= t < SPEECH_END:
                codes.append(t - SPEECH_OFFSET)
        gen_time = time.time() - start

        t1 = time.time()
        wav = codec.decode(global_embedding=voice,
                           content_token_indices=torch.tensor(codes))
        codec_time = time.time() - t1

        secs = len(codes) / 25
        rtf = (gen_time + codec_time) / secs
        rtfs.append(rtf)

        path = f"{WORK}/{name}_{i}.wav"
        sf.write(path, wav.squeeze().float().numpy(), SR)
        print(f"{text[:28]:28s} | {secs:.1f}s audio | model {gen_time:.2f}s "
              f"({len(codes)/gen_time:.1f} tok/s) + codec {codec_time:.2f}s | RTF {rtf:.2f}")
        display(Audio(path))

    summary[name] = sum(rtfs) / len(rtfs)
    del llm


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/13.8M [00:00<?, ?B/s]

...

This is a test sentence in E | 1.8s audio | model 2.27s (19.8 tok/s) + codec 0.14s | RTF 1.34


प्लान तो बढ़िया है, but wait | 2.7s audio | model 3.56s (18.8 tok/s) + codec 0.17s | RTF 1.39



===== Q4 =====
नमस्ते, आप कैसे हैं? <happy> | 2.6s audio | model 2.67s (23.9 tok/s) + codec 0.17s | RTF 1.11


This is a test sentence in E | 1.6s audio | model 2.02s (20.3 tok/s) + codec 0.14s | RTF 1.32


प्लान तो बढ़िया है, but wait | 2.9s audio | model 2.95s (24.7 tok/s) + codec 0.18s | RTF 1.07


In [10]:
print("average RTF")
for name, rtf in summary.items():
    verdict = "PASS - faster than real time" if rtf < 1 else "too slow"
    print(f"  {name.upper()}: {rtf:.2f}   {verdict}")


average RTF
  Q8: 1.41   too slow
  Q4: 1.17   too slow


In [12]:
from IPython.display import FileLink, display

display(FileLink("indic-mio-q8.gguf"))   # ~640 MB
display(FileLink("indic-mio-q4.gguf"))   # ~380 MB

/kaggle/working/indic-mio-q8.gguf

/kaggle/working/indic-mio-q4.gguf